# ARC-AGI-2 custom tensor playground

This notebook is a manual modeling sandbox. Pick any public ARC task, inspect any input/output grid, represent it as an `H × W × 10` tensor, apply your own operations, render the result, and score your submitted tensor against the known target.

ARC itself stores a grid as a 2D matrix of integers `0..9`. Here we encode that losslessly as 10 one-hot channels:

`grid[y, x] = c  <=>  tensor[y, x, c] = 1`

Your custom operation does **not** have to preserve one-hot values. It may return probabilities, logits, or arbitrary real channel scores; evaluation decodes with `argmax(axis=-1)`.

## 0. One-time setup

From the repository root:

```bash
make setup
make data
```

The notebook uses the official public ARC-AGI-2 task repository cloned into `data/ARC-AGI-2`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from arc_agi2_fun.data import load_official_split
from arc_agi2_fun.tensors import (
    NUM_COLORS,
    evaluate_tensor,
    get_task_grid,
    grid_to_tensor,
    tensor_to_grid,
)
from arc_agi2_fun.visualize import plot_grid, plot_task

DATA_DIR = ROOT / 'data' / 'ARC-AGI-2'
print('repo:', ROOT)
print('data:', DATA_DIR)

## 1. Load a split and choose any task

Use `training` while developing your approach. Treat `evaluation` as an outer holdout when you want a more honest measurement.

In [ ]:
SPLIT = 'training'  # 'training' or 'evaluation'

tasks = load_official_split(DATA_DIR, SPLIT)
task_ids = list(tasks)
print(f'{len(task_ids)} tasks loaded from {SPLIT}')
print('first 20 ids:', task_ids[:20])

for i in range(3,4):
    TASK_ID = task_ids[i]  # replace with any id printed above
    task = tasks[TASK_ID]
    print('selected:', TASK_ID)
    print('train examples:', len(task.train))
    print('test examples:', len(task.test_inputs))

    plot_task(task)
    plt.show()

In [ ]:
from ops import init_env
import numpy as np
import pandas as pd

# --------------------------------------------------
# Initialize the 5-component environment
# --------------------------------------------------

GP_meta, GP_X, SP_meta, SP_X, ST = init_env(task.train)


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def show_meta(meta, name):
    print(f"\n{name}")
    display(
        pd.DataFrame(meta.rows()).set_index("gidx")
    )


def summarize_X(meta, X, name):
    rows = []

    for gidx in range(len(X)):
        for sample_idx in range(X.sample_count):
            value = X[gidx, sample_idx]
            arr = np.asarray(value)

            rows.append({
                "gidx": gidx,
                "sample": sample_idx,
                "op": meta.op[gidx],
                "source": meta.source[gidx],
                "dims": meta.dims[gidx],
                "shape": arr.shape,
                "dtype": str(arr.dtype),
            })

    print(f"\n{name}")
    display(pd.DataFrame(rows))


# --------------------------------------------------
# Metadata
# --------------------------------------------------

show_meta(GP_meta, "GP_meta")
show_meta(SP_meta, "SP_meta")


# --------------------------------------------------
# Instantiated gene summaries
# --------------------------------------------------

summarize_X(GP_meta, GP_X, "GP_X")
summarize_X(SP_meta, SP_X, "SP_X")


# --------------------------------------------------
# Solution Tree
# --------------------------------------------------

print("\nST")
display(
    pd.DataFrame(ST.rows()).set_index("sp_gidx")
)


# --------------------------------------------------
# Basic environment dimensions
# --------------------------------------------------

print("\nEnvironment")
print("GP_X shape [genes, samples]:", GP_X.shape)
print("SP_X shape [genes, samples]:", SP_X.shape)
print("ST roots:", ST.roots)


# --------------------------------------------------
# Direct inspection examples
# --------------------------------------------------

sample_idx = 0

print("\nSample 0")

print("\nGP gene 0 — input:")
print(GP_X[0, sample_idx])

print("\nGP gene 1 — input shape:")
print(GP_X[1, sample_idx])

print("\nGP gene 2 — input colors:")
print(GP_X[2, sample_idx])

print("\nGP gene 3 — input color presence:")
print(GP_X[3, sample_idx])


print("\nSP gene 0 — output:")
print(SP_X[0, sample_idx])

print("\nSP gene 1 — output shape:")
print(SP_X[1, sample_idx])

print("\nSP gene 2 — output colors:")
print(SP_X[2, sample_idx])

print("\nSP gene 3 — output color presence:")
print(SP_X[3, sample_idx])

In [ ]:
from ops import (
    init_env,
    GP_generate,
    SP_generate,
    OP_REGISTRY,
    valid_generation,
    generation_invalid_reason,
    gene_atomic_dtypes,
)

import numpy as np
import pandas as pd


# ============================================================
# Initialize
# ============================================================

GP_meta, GP_X, SP_meta, SP_X, ST = init_env(task.train)


# ============================================================
# Helpers
# ============================================================

def show_meta(meta, title):
    print(f"\n=== {title} ===")
    display(
        pd.DataFrame(meta.rows()).set_index("gidx")
    )


def show_registry():
    rows = []

    for name, info in OP_REGISTRY.items():
        rows.append({
            "op": name,
            "full_partition": info.full_partition,
            "outputs": info.output_count,
            "dims >": info.min_dims_exclusive,
            "atomic_dtypes": (
                None
                if info.atomic_dtypes is None
                else [str(x) for x in info.atomic_dtypes]
            ),
            "parameter_sampler": info.parameter_sampler is not None,
        })

    print("\n=== OP REGISTRY ===")
    display(pd.DataFrame(rows).set_index("op"))


def show_generation_space(meta, X, title):
    rows = []

    for gidx in range(len(X)):
        for op_name, op_info in OP_REGISTRY.items():

            # Current initial ops have no parameters.
            params = {}

            valid = valid_generation(
                meta,
                X,
                op_info,
                gidx,
                params=params,
            )

            rows.append({
                "source_gidx": gidx,
                "source_op": meta.op[gidx],
                "dims": meta.dims[gidx],
                "dtype": [
                    str(x)
                    for x in gene_atomic_dtypes(X, gidx)
                ],
                "candidate_op": op_name,
                "valid": valid,
                "reason": (
                    None
                    if valid
                    else generation_invalid_reason(
                        meta,
                        X,
                        op_info,
                        gidx,
                        params=params,
                    )
                ),
            })

    print(f"\n=== {title} GENERATION SPACE ===")
    display(pd.DataFrame(rows))


# ============================================================
# BEFORE GENERATION
# ============================================================

show_registry()

show_meta(GP_meta, "GP_meta BEFORE")
show_meta(SP_meta, "SP_meta BEFORE")

show_generation_space(
    GP_meta,
    GP_X,
    "GP",
)

show_generation_space(
    SP_meta,
    SP_X,
    "SP",
)


# ============================================================
# GENERATE
# ============================================================

new_gp = GP_generate(
    GP_meta,
    GP_X,
    n_new_genes=10,
    rng=42,
)

new_sp = SP_generate(
    SP_meta,
    SP_X,
    ST,
    rng=42,
)


print("\nNew GP genes:", new_gp)
print("New SP genes:", new_sp)


# ============================================================
# AFTER GENERATION
# ============================================================

show_meta(GP_meta, "GP_meta AFTER")
show_meta(SP_meta, "SP_meta AFTER")

print("\nGP_X shape [genes, samples]:", GP_X.shape)
print("SP_X shape [genes, samples]:", SP_X.shape)


# ============================================================
# Inspect newly generated values
# ============================================================

sample_idx = 0

print("\n=== NEW GP VALUES — SAMPLE 0 ===")

for gidx in new_gp:
    print(
        f"\ngidx={gidx}"
        f" | op={GP_meta.op[gidx]}"
        f" | source={GP_meta.source[gidx]}"
        f" | dims={GP_meta.dims[gidx]}"
        f" | params={GP_meta.params[gidx]}"
    )

    print(GP_X[gidx, sample_idx])


print("\n=== NEW SP VALUES — SAMPLE 0 ===")

for gidx in new_sp:
    print(
        f"\ngidx={gidx}"
        f" | op={SP_meta.op[gidx]}"
        f" | source={SP_meta.source[gidx]}"
        f" | dims={SP_meta.dims[gidx]}"
        f" | params={SP_meta.params[gidx]}"
    )

    print(SP_X[gidx, sample_idx])


# ============================================================
# Updated Solution Tree
# ============================================================

print("\n=== ST ===")

display(
    pd.DataFrame(ST.rows()).set_index("sp_gidx")
)

In [ ]:
for i in range(8):
    print((GP_X[i][0]))

In [ ]:
def inspect_loss(tree):
    h = tree["shape"]["h"].value
    w = tree["shape"]["w"].value
    M = tree["composite"].value

    print(
        f"Height: {h.predicted} -> {h.target} "
        f"{'✓' if h.resolved else '✗'}"
    )

    print(
        f"Width:  {w.predicted} -> {w.target} "
        f"{'✓' if w.resolved else '✗'}"
    )

    print("\nColor transitions:")

    for pred in range(11):
        for target in range(11):
            count = M[pred, target]

            if count:
                print(
                    f"{pred:>2} -> {target:<2}: {count}"
                )

    print("\nFully resolved:", tree.resolved)

In [ ]:
from ops import GP
from ops.loss import loss_resolution, inspect_loss

from importlib import reload
import ops.grid_ops as grid_ops
import ops.GP as GP
reload(GP)
reload(grid_ops)

# Build GP structure from the task's training examples
gpmat = GP.init_gp_mat(task.train)

# Compare all training inputs against their corresponding outputs
tree = loss_resolution(
    gpmat.input,
    gpmat.output,
)

# Human-readable overview
inspect_loss(tree)

In [ ]:
print("\nHEIGHT TRANSITIONS")
print(tree.h.matrix)

print("\nWIDTH TRANSITIONS")
print(tree.w.matrix)

print("\nCOMPOSITE COLOR TRANSITIONS")
print(tree.composite.matrix)

In [ ]:
print((gpmat.data[0])[3])

In [ ]:


from ops.grid_ops import grid_dissection

d = grid_dissection(task.train[0].input)

In [ ]:
print((d[1].shape))

In [ ]:
from importlib import reload
import ops.grid_ops as grid_ops
import ops.GP as GP
reload(GP)
reload(grid_ops)

gpmat = GP.init_gp_mat(task.train)

#print(type(gpmat.data[0]))
#print((gpmat.data[0]))
for i in range(len(task.train)):
    #plot_grid(gpmat.input[i])
    #plot_grid(gpmat.data[i][0])
    #plot_grid(gpmat.output[i])
    #print(len(gpmat.data[i]))
    pass

print(gpmat.status)
GP.gp_fill_status(gpmat)
print(gpmat.status)



In [ ]:
from ops.grid_ops import reconstruct_grid

grid1 = reconstruct_grid(d)

plot_grid(task.train[0].input)

In [ ]:
plot_grid(task.train[0].output)

## 2. Render the entire task

Top row shows inputs; bottom row shows the matching outputs when known. Public ARC-AGI-2 files include outputs so you can use them for research/evaluation.

## 3. Pick one example and inspect either the before or after grid

`SECTION='train'` selects a demonstration pair. `SECTION='test'` selects a public test pair. `SIDE='input'` is the before grid and `SIDE='output'` is the after/solution grid.

In [ ]:
SECTION = 'train'   # 'train' or 'test'
for i in range(5):
    EXAMPLE_INDEX = i
    SIDE = 'input'        # 'input' or 'output'

    selected_grid = get_task_grid(
        task,
        section=SECTION,
        index=EXAMPLE_INDEX,
        side=SIDE,
    )
    selected_tensor = grid_to_tensor(selected_grid)

    print('2D grid shape:', np.asarray(selected_grid).shape)
    print('3D tensor shape:', selected_tensor.shape)
    print('channels:', NUM_COLORS)

    plot_grid(selected_grid, f'{TASK_ID} | {SECTION}[{EXAMPLE_INDEX}] | {SIDE}')
    plt.show()

    import numpy as np
    from ops.alpha_ops import survey_abs_kernels

    # selected_grid is your normal H x W ARC integer grid
    x = np.asarray(selected_grid, dtype=float)

    responses = survey_abs_kernels(
        x,
        kernel="self",
        plot=True,
    )

    # Example: inspect one raw response matrix
    #responses

## 4. Inspect the actual 3D representation

For every `(y, x)` cell, the length-10 vector identifies its color. A color-3 pixel is `[0,0,0,1,0,0,0,0,0,0]`.

In [ ]:
Y = 0
X = 0

print('integer color:', selected_grid[Y][X])
print('channel vector:', selected_tensor[Y, X])
print('active channel:', int(np.argmax(selected_tensor[Y, X])))

selected_tensor

## 5. Build a modeling pair

For actual transformation work, use the input tensor as `model_input` and the corresponding output tensor as `target_tensor`.

In [ ]:
PAIR_SECTION = 'train'  # switch to 'test' for public held-out examples
PAIR_INDEX = 0

input_grid = get_task_grid(task, section=PAIR_SECTION, index=PAIR_INDEX, side='input')
target_grid = get_task_grid(task, section=PAIR_SECTION, index=PAIR_INDEX, side='output')

model_input = grid_to_tensor(input_grid)
target_tensor = grid_to_tensor(target_grid)

print('input tensor:', model_input.shape)
print('target tensor:', target_tensor.shape)

## 6. Your custom operation sandbox

Edit **only this function** while experimenting. Input is an `H × W × 10` NumPy array. Output must also be a 3D array with 10 channels, although height and width may change when the inferred ARC rule changes output dimensions.

Useful spatial operations preserve the channel axis:

```python
np.rot90(x, 1, axes=(0, 1))
x[:, ::-1, :]       # horizontal flip
x[::-1, :, :]       # vertical flip
```

You can also manipulate channels, build masks, detect objects, create a brand-new tensor, or eventually call a learned model.

In [ ]:
def my_operation(x: np.ndarray) -> np.ndarray:
    # ---- YOUR MODEL / TRANSFORMATION GOES HERE ----
    y = x.copy()

    # Examples to try one at a time:
    # y = np.rot90(x, 1, axes=(0, 1))
    # y = x[:, ::-1, :]
    # y = x[::-1, :, :]

    return y

submitted_tensor = my_operation(model_input.copy())
print('submitted tensor shape:', submitted_tensor.shape)

## 7. Render your submitted 3D tensor

Rendering first decodes the 10 channels by `argmax`, exactly like the evaluator.

In [ ]:
submitted_grid = tensor_to_grid(submitted_tensor)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_grid(input_grid, 'input', ax=axes[0])
plot_grid(submitted_grid, 'your submitted output', ax=axes[1])
plot_grid(target_grid, 'true output', ax=axes[2])
plt.show()

## 8. Evaluate the submitted tensor against the solution

Competition success is **exact decoded-grid equality**, including output dimensions. Cell accuracy is provided only as a useful diagnostic while building your model.

In [ ]:
evaluation = evaluate_tensor(submitted_tensor, target_tensor)
pd.DataFrame([evaluation.to_dict()])

## 9. Direct matrix experimentation

You do not need to use `my_operation`. Any `H × W × 10` array can be submitted to the evaluator. For example, this is useful if a neural model returns logits.

In [ ]:
# Example placeholder for arbitrary model scores/logits.
# Replace this with the output of whatever modeling approach you build.
model_scores = submitted_tensor.astype(np.float32)

result = evaluate_tensor(model_scores, target_tensor)
print(result)
print('decoded prediction:')
print(np.asarray(tensor_to_grid(model_scores)))

## 10. Modeling contract

This notebook does not impose a modeling philosophy. Your operation can become convolutional, object-based, symbolic, grammatical, genetic, MCTS-driven, or hybrid.

The fixed contract is:

`ARC grid -> H×W×10 tensor -> your operation/model -> H'×W'×10 tensor -> exact evaluation`.

## 11. Alpha operations: absolute neighborhood surveys

The first notebook-local analytical operation lives in `ops/alpha_ops.py`. `survey_abs_kernels` compares every pixel/vector with selected neighbors using absolute differences. With `kernel='all'`, it evaluates the full default kernel bank and returns a dictionary of raw `H × W` response maps.

In [ ]:
from ops.alpha_ops import DEFAULT_ABS_KERNELS, survey_abs_kernels

print('available alpha kernels:')
for name, kernel in DEFAULT_ABS_KERNELS.items():
    print(f'\n{name}:\n{kernel}')

In [ ]:
# Run the full survey directly on the H × W × 10 ARC tensor.
alpha_responses = survey_abs_kernels(
    model_input,
    kernel='all',
    plot=True,
)

# Raw floating response maps remain available for your own modeling.
alpha_responses['left']

A custom kernel can be supplied directly as a 1D or 2D list/array. For example, `[1, 1, 0]` is the same center-anchored left survey as the named `left` kernel.

In [ ]:
custom_left = survey_abs_kernels(
    model_input,
    kernel=[1, 1, 0],
    plot=True,
)
custom_left